# ML Laboratuvarı · Modül 3, Ünite 9: Enflasyonu öngörmek

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 3

Veri: 13 aylık ABD serisi (FRED). Kurulum hücresi `G` adlı aylık bir tablo hazırlar (indeks: tarih). Sütunlar: `enf` (yıllık enflasyon, %), `aylik` (aylık enflasyon, %), 7 serinin yıllık değişimi (`CPILFESL`, `PPIACO`, `M2SL`, `WTISPLC`, `AHETPI`, `PAYEMS`, `INDPRO`), 4 serinin düzeyi (`UNRATE`, `FEDFUNDS`, `T10Y3MM`, `UMCSENT`) ve `HOUST` (log). Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

> Not: Ekim 2025'in TÜFE verisi yok (ABD federal hükümetinin kapanması). Eksik ayları doldurma; içeren satırları dışarıda bırak. Basitlik için “bu ayın verisi ay sonunda biliniyor” varsayıyoruz; gerçekte TÜFE bir sonraki ayın ortasında yayımlanır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, base64, urllib.request, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold
warnings.filterwarnings("ignore")

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

kodlar = ['CPIAUCSL', 'CPILFESL', 'PPIACO', 'M2SL', 'WTISPLC', 'AHETPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'FEDFUNDS', 'T10Y3MM', 'UMCSENT', 'HOUST']
ham = pd.concat([fred_indir(k).set_index("tarih") for k in kodlar], axis=1).sort_index().asfreq("MS")
G = pd.DataFrame(index=ham.index)
G["enf"] = ham["CPIAUCSL"].pct_change(12, fill_method=None) * 100     # yıllık enflasyon (%)
G["aylik"] = ham["CPIAUCSL"].pct_change(1, fill_method=None) * 100    # aylık enflasyon (%)
for k in ["CPILFESL", "PPIACO", "M2SL", "WTISPLC", "AHETPI", "PAYEMS", "INDPRO"]:
    G[k] = ham[k].pct_change(12, fill_method=None) * 100              # yıllık değişim (%)
for k in ["UNRATE", "FEDFUNDS", "T10Y3MM", "UMCSENT"]:
    G[k] = ham[k]                                                      # düzey
G["HOUST"] = np.log(ham["HOUST"])                                      # konut başlangıcı (log)
_G = G.copy()
BASLA, KESIM = pd.Timestamp("1984-01-01"), pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _referans():
    oz = {}
    for k in _G.columns:
        if k != "aylik":
            for g in range(3):
                oz[f"{k}_g{g}"] = _G[k].shift(g)
    for g in range(12):
        oz[f"aylik_g{g}"] = _G["aylik"].shift(g)
    X = pd.DataFrame(oz)
    t = pd.concat([X, _G["enf"].shift(-1).rename("hedef")], axis=1).loc[BASLA:].dropna()
    e, s = t.loc[:KESIM - pd.Timedelta(days=1)], t.loc[KESIM:]
    c = list(X.columns)
    kh = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
    sc = StandardScaler().fit(e[c])
    ekk = LinearRegression().fit(sc.transform(e[c]), e["hedef"])
    las = LassoCV(alphas=np.logspace(-4, 0.5, 60), cv=TimeSeriesSplit(5), max_iter=50000).fit(sc.transform(e[c]), e["hedef"])
    return dict(n_test=len(s), saf=kh(s["enf_g0"], s["hedef"]), ekk=kh(ekk.predict(sc.transform(s[c])), s["hedef"]),
                lasso=kh(las.predict(sc.transform(s[c])), s["hedef"]))

def kontrol_A(tablo, egitim, test, kok_hko_saf, kok_hko_ekk):
    if "hedef" not in tablo.columns:
        print("❌ 'tablo' içinde 'hedef' adlı bir sütun bulunamadı."); return
    h = tablo["hedef"]
    if np.allclose(h.values, _G["enf"].reindex(h.index).values, equal_nan=True):
        print("❌ Hedef kaydırılmamış: 'hedef' bu ayın enflasyonu. Gelecek ayın değeri için shift(-1) gerekir (hedef sızıntısı)."); return
    if not np.allclose(h.values, _G["enf"].shift(-1).reindex(h.index).values, equal_nan=True):
        print("❌ 'hedef' gelecek ayın yıllık enflasyonu (enf.shift(-1)) ile uyuşmuyor. Hedef farklı kurulmuş ya da eksik aylar doldurulmuş olabilir."); return
    if tablo.isna().sum().sum() > 0:
        print("❌ Tabloda eksik değer var; eksik ayları içeren satırlar dışarıda bırakılmalı (dropna)."); return
    if len(tablo.columns) - 1 != 51:
        print(f"❌ 51 özellik bekleniyordu, {len(tablo.columns) - 1} var. Tarifi yeniden oku: 13 seri × 3 gecikme + aylık enflasyonun 12 gecikmesi."); return
    if not egitim.index.is_monotonic_increasing or not test.index.is_monotonic_increasing or egitim.index.max() >= KESIM or test.index.min() < KESIM:
        print("❌ Bölme zamana göre değil: eğitim 2015 öncesi, test 2015 ve sonrası olmalı; karıştırma olmamalı."); return
    r = _referans()
    if len(test) > r["n_test"]:
        print("❌ Test döneminde beklenenden fazla ay var: eksik aylar doldurulmuş olabilir. Doldurma; dışarıda bırak."); return
    if egitim.index.min() > BASLA + pd.Timedelta(days=40):
        print("❌ Eğitim verisi 1984'ten başlamıyor."); return
    if abs(float(kok_hko_saf) - r["saf"]) > 0.005:
        print("❌ Saf öngörünün kök HKO'su beklenenden farklı. Öngörü 'bu ayın yıllık enflasyonu' (enf_g0) mı, hedefle mi karşılaştırıldı?"); return
    if abs(float(kok_hko_ekk) - r["ekk"]) > 0.02:
        print("❌ EKK'nın test kök HKO'su beklenenden farklı. 51 özelliğin hepsi kullanıldı mı? Model yalnızca eğitimle mi kuruldu?"); return
    print(f"✅ Tüm kontroller geçti (saf ≈ {r['saf']:.3f}, EKK ≈ {r['ekk']:.3f}). Tamamlama kodun: " + _k("TUwzLUVORi00S1Fa"))

def kontrol_B(lasso, kok_hko_lasso, secilenler):
    if isinstance(getattr(lasso, "cv", None), KFold) or not isinstance(getattr(lasso, "cv", None), TimeSeriesSplit):
        print("❌ λ zamana saygılı çapraz doğrulamayla seçilmemiş. cv=TimeSeriesSplit(5) kullan; karıştırılmış KFold geleceği gösterir."); return
    r = _referans()
    if abs(float(kok_hko_lasso) - r["lasso"]) > 0.01:
        print("❌ Lasso'nun test kök HKO'su beklenenden farklı. Ölçekleyici yalnızca eğitim verisiyle mi kuruldu? alphas ızgarası ve max_iter isteğe uygun mu?"); return
    nz = int(np.sum(np.abs(lasso.coef_) > 0))
    if len(secilenler) != nz:
        print(f"❌ 'secilenler' {len(secilenler)} öğe içeriyor, ama sıfırdan farklı katsayı sayısı {nz}. Negatif katsayıları da say (coef_ != 0)."); return
    if "aylik_g11" not in secilenler:
        print("ℹ️ Bilgi: aylik_g11 (baz etkisi) listede yok; özellik adlarını kontrol et.")
    print(f"✅ Tüm kontroller geçti (Lasso ≈ {r['lasso']:.3f}, saf ≈ {r['saf']:.3f}). Tamamlama kodun: " + _k("TUwzLUxTTy03WFBE"))

print("G tablosu:", G.shape)
G.tail(3).round(2)

## Görev A — Problem ve kıyas ölçütü

1. **Özellikler:** `G`'nin `aylik` dışındaki her sütunu için 0, 1 ve 2 ay gecikmeli değerler (`{sütun}_g0`, `{sütun}_g1`, `{sütun}_g2`) ve `aylik`'in 0–11 ay gecikmeli değerleri (`aylik_g0` … `aylik_g11`): toplam 51.
2. **Hedef:** gelecek ayın yıllık enflasyonu, `hedef` adlı sütun: `G["enf"].shift(-1)`.
3. Özellikleri ve hedefi `tablo` adlı bir tabloda birleştir, `1984-01-01`'den başlat ve eksik değerli satırları at.
4. **Zamana göre böl:** `2015-01-01` öncesi `egitim`, sonrası `test`. Özellik adlarını `ozellikler` listesine yaz.
5. Test döneminde **saf öngörünün** (`enf_g0`) ve eğitim verisiyle standartlaştırıp kurduğun **51 değişkenli EKK**'nın kök HKO'sunu hesapla: `kok_hko_saf`, `kok_hko_ekk`.

**Denetim soruları:** Hedef `shift(-1)` mi? Özelliklerde negatif kaydırma var mı? Bölmede karıştırma var mı? Eksik aylar doldurulmuş mu?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(tablo, egitim, test, kok_hko_saf, kok_hko_ekk)

## Görev B — Lasso

1. `StandardScaler`'ı **yalnızca** `egitim[ozellikler]` ile kur; test verisine yalnızca `transform` uygula.
2. `LassoCV(alphas=np.logspace(-4, 0.5, 60), cv=TimeSeriesSplit(5), max_iter=50000)` ile λ'yı seç; modeli `lasso` değişkenine ata.
3. Test kök HKO'sunu `kok_hko_lasso`'ya ata ve saf öngörüyle karşılaştır.
4. Katsayısı sıfırdan farklı özellikleri `secilenler` listesine yaz; mutlak değere göre ilk 8'ini katsayılarıyla yazdır.

**Denetim soruları:** Ölçekleyici neyle kuruldu? `cv` ne? Seçilen değişkenler iktisadi olarak ne anlatıyor (baz etkisi: `aylik_g11`)?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(lasso, kok_hko_lasso, secilenler)

## Denetim notların

- Gemini hedefi ve bölmeyi nasıl kurdu? Değiştirmen gerekti mi?
- Ölçekleme ve çapraz doğrulama nasıl yapıldı?
- Lasso saf öngörüyü ne kadar geçiyor? Hangi değişkenler seçildi, ne anlama geliyor?